# Exercícios de Regressão Logística — Resolução

Resolução dos exercícios do **Guia de Estudos - Prova.md**, seguindo o mesmo padrão de código
usado na Aula 5 (função sigmoide e entropia cruzada implementadas com Numpy puro).

In [1]:
# =======================
# funções do modelo
# =======================
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))


## Exercício 1 — Classificando clientes

Um banco usa $z = -3 + 0{,}8 \cdot \text{renda (em milhares)} - 0{,}5 \cdot \text{nº de dívidas ativas}$
para prever se aprova (1) ou não (0) um crédito.

| Cliente | Renda | Dívidas |
|---|---|---|
| P | 5 | 1 |
| Q | 3 | 0 |
| R | 4 | 3 |

**Pergunta:** calcule $z$, a probabilidade $\sigma(z)$ e a classificação final de cada cliente.

In [2]:
beta0, beta1, beta2 = -3, 0.8, -0.5  # coeficientes do modelo já treinado (fornecidos pelo exercício)


#### Obtendo os dados dos clientes — duas formas equivalentes

In [3]:
# Forma 1: dados digitados direto no código
nomes = ["P", "Q", "R"]
dados_clientes = np.array([
    [5, 1],   # renda, dívidas do cliente P
    [3, 0],   # renda, dívidas do cliente Q
    [4, 3],   # renda, dívidas do cliente R
])

print("nomes:", nomes)
print("dados_clientes:\n", dados_clientes)


nomes: ['P', 'Q', 'R']
dados_clientes:
 [[5 1]
 [3 0]
 [4 3]]


In [4]:
# Forma 2: carregando os MESMOS dados de um arquivo CSV
# arquivo "exercicio1_clientes.csv":
#   cliente,renda,dividas
#   P,5,1
#   Q,3,0
#   R,4,3
# usecols=(1, 2) pula a 1ª coluna (nome, é texto, não dá pra carregar com np.loadtxt)
dados_clientes = np.loadtxt("exercicio1_clientes.csv", delimiter=",", skiprows=1, usecols=(1, 2))
nomes = ["P", "Q", "R"]  # nomes continuam à parte, pois não são um dado numérico

print("nomes:", nomes)
print("dados_clientes:\n", dados_clientes)


nomes: ['P', 'Q', 'R']
dados_clientes:
 [[5. 1.]
 [3. 0.]
 [4. 3.]]


In [5]:
for nome, (renda, dividas) in zip(nomes, dados_clientes):
    z = beta0 + beta1 * renda + beta2 * dividas
    prob = sigmoid(z)
    classe = 1 if prob >= 0.5 else 0
    resultado_texto = "APROVADO" if classe == 1 else "NEGADO"
    print(f"Cliente {nome}: z={z:.2f} | probabilidade={prob:.4f} ({prob * 100:.1f}%) | crédito {resultado_texto}")


Cliente P: z=0.50 | probabilidade=0.6225 (62.2%) | crédito APROVADO
Cliente Q: z=-0.60 | probabilidade=0.3543 (35.4%) | crédito NEGADO
Cliente R: z=-1.30 | probabilidade=0.2142 (21.4%) | crédito NEGADO


### Respostas — Exercício 1

Resultados calculados na célula acima:

- **P:** $z=0{,}5 \to \sigma(0{,}5)\approx0{,}622$ (62,2%) → **crédito aprovado**
- **Q:** $z=-0{,}6 \to \sigma(-0{,}6)\approx0{,}354$ (35,4%) → **crédito negado**
- **R:** $z=-1{,}3 \to \sigma(-1{,}3)\approx0{,}214$ (21,4%) → **crédito negado**

## Exercício 2 — Comparando o custo (log loss) de dois modelos

Um modelo previu as probabilidades $p=[0{,}8,\ 0{,}3,\ 0{,}6]$ para 3 exemplos cujos valores reais
são $y=[1,\ 0,\ 1]$. Um segundo modelo previu $p=[0{,}95,\ 0{,}1,\ 0{,}9]$ para os mesmos $y$.

**Perguntas:** a) calcule o log loss de cada modelo; b) qual modelo é melhor? Por quê?

In [6]:
def compute_cost(y, h):
    """Calcula o custo (log loss / entropia cruzada binária)."""
    m = len(y)
    epsilon = 1e-8
    cost = -(1 / m) * np.sum(
        y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon)
    )
    return cost


#### Obtendo y e as probabilidades — duas formas equivalentes

In [7]:
# Forma 1: dados digitados direto no código
y = np.array([1, 0, 1])
p_modelo1 = np.array([0.8, 0.3, 0.6])
p_modelo2 = np.array([0.95, 0.1, 0.9])

print("y:", y)
print("p_modelo1:", p_modelo1)
print("p_modelo2:", p_modelo2)


y: [1 0 1]
p_modelo1: [0.8 0.3 0.6]
p_modelo2: [0.95 0.1  0.9 ]


In [8]:
# Forma 2: carregando os MESMOS dados de um arquivo CSV
# arquivo "exercicio2_probabilidades.csv":
#   y_real,p_modelo1,p_modelo2
#   1,0.8,0.95
#   0,0.3,0.1
#   1,0.6,0.9
dados = np.loadtxt("exercicio2_probabilidades.csv", delimiter=",", skiprows=1)
y = dados[:, 0]
p_modelo1 = dados[:, 1]
p_modelo2 = dados[:, 2]

print("y:", y)
print("p_modelo1:", p_modelo1)
print("p_modelo2:", p_modelo2)


y: [1. 0. 1.]
p_modelo1: [0.8 0.3 0.6]
p_modelo2: [0.95 0.1  0.9 ]


In [9]:
custo_modelo1 = compute_cost(y, p_modelo1)
custo_modelo2 = compute_cost(y, p_modelo2)

print(f"Custo (log loss) do Modelo 1: {custo_modelo1:.4f}")
print(f"Custo (log loss) do Modelo 2: {custo_modelo2:.4f}")

melhor_modelo = "Modelo 1" if custo_modelo1 < custo_modelo2 else "Modelo 2"
print(f"\n--> O {melhor_modelo} é melhor: quanto menor o custo, mais as probabilidades previstas")
print("    se aproximam dos valores reais.")


Custo (log loss) do Modelo 1: 0.3635
Custo (log loss) do Modelo 2: 0.0873

--> O Modelo 2 é melhor: quanto menor o custo, mais as probabilidades previstas
    se aproximam dos valores reais.


### Respostas — Exercício 2

**a)** Log loss do Modelo 1 ≈ **0,3635** (calculado na célula acima).

**b)** Log loss do Modelo 2 ≈ **0,0873** (calculado na célula acima).

**c)** O **Modelo 2** é melhor: seu log loss (≈0,087) é bem menor que o do Modelo 1 (≈0,364).
Quanto menor o custo, mais as probabilidades previstas se aproximam dos valores reais.